# 03 Feature Engineering

This notebook creates the engineered feature dataset for the MetroPT-3 predictive maintenance framework.

It uses the cleaned dataset from:

`data/processed/metropt3_cleaned.csv`

It creates:

- 15-minute and 60-minute rolling features
- 5-minute, 15-minute and 60-minute lag features
- rate-of-change features
- pressure-difference features
- feature summary tables
- missing-value diagnostics

Final output:

`data/processed/metropt3_features.csv`

In [1]:
# 1. Import libraries and confirm environment

import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Python executable:", sys.executable)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)

Python executable: c:\Users\user\Desktop\metropt_predictive_maintenance_starter\.venv\Scripts\python.exe
pandas: 3.0.5
numpy: 2.5.2


## 1. Set project paths

This block detects whether the notebook is being run from the project root or from the `notebooks/` folder.

In [2]:
# 2. Define project paths

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
OUTPUT_FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_TABLES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

cleaned_path = PROCESSED_DATA_DIR / "metropt3_cleaned.csv"
features_path = PROCESSED_DATA_DIR / "metropt3_features.csv"

print("Project root:", PROJECT_ROOT)
print("Cleaned dataset path:", cleaned_path)
print("Feature dataset output path:", features_path)

Project root: c:\Users\user\Desktop\metropt_predictive_maintenance_starter\metropt_predictive_maintenance_starter
Cleaned dataset path: c:\Users\user\Desktop\metropt_predictive_maintenance_starter\metropt_predictive_maintenance_starter\data\processed\metropt3_cleaned.csv
Feature dataset output path: c:\Users\user\Desktop\metropt_predictive_maintenance_starter\metropt_predictive_maintenance_starter\data\processed\metropt3_features.csv


## 2. Load cleaned dataset

The cleaned dataset should already have been produced by `02_preprocessing_eda.ipynb`.

In [3]:
# 3. Load cleaned dataset

if not cleaned_path.exists():
    raise FileNotFoundError(
        f"Cleaned dataset not found at {cleaned_path}. "
        "Run 02_preprocessing_eda.ipynb first."
    )

df = pd.read_csv(cleaned_path)

print("Loaded cleaned dataset.")
print("Shape:", df.shape)
df.head()

Loaded cleaned dataset.
Shape: (1516948, 17)


,Unnamed: 0,timestamp,TP2,TP3,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,Caudal_impulses
0,0,2020-02-01 00:00:00,-0.012,9.358,9.340,-0.024,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
1,10,2020-02-01 00:00:10,-0.014,9.348,9.332,-0.022,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
2,20,2020-02-01 00:00:19,-0.012,9.338,9.322,-0.022,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
3,30,2020-02-01 00:00:29,-0.012,9.328,9.312,-0.022,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0
4,40,2020-02-01 00:00:39,-0.012,9.318,9.302,-0.022,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,1.0


## 3. Detect timestamp column and sort chronologically

Feature engineering must preserve time order to avoid temporal leakage.

In [4]:
# 4. Detect timestamp column

possible_timestamp_cols = ["timestamp", "Timestamp", "time", "Time", "datetime", "Datetime", "date", "Date"]

timestamp_col = None
for col in possible_timestamp_cols:
    if col in df.columns:
        timestamp_col = col
        break

if timestamp_col is None:
    print("Available columns:", df.columns.tolist())
    raise ValueError("Timestamp column not detected. Please set timestamp_col manually.")

df[timestamp_col] = pd.to_datetime(df[timestamp_col], errors="coerce")
invalid_timestamps = df[timestamp_col].isna().sum()

df = df.dropna(subset=[timestamp_col])
df = df.sort_values(timestamp_col).reset_index(drop=True)

print("Timestamp column:", timestamp_col)
print("Invalid timestamps removed:", invalid_timestamps)
print("Start:", df[timestamp_col].min())
print("End:", df[timestamp_col].max())
print("Shape after timestamp check:", df.shape)

Timestamp column: timestamp
Invalid timestamps removed: 0
Start: 2020-02-01 00:00:00
End: 2020-09-01 03:59:50
Shape after timestamp check: (1516948, 17)


## 4. Select numerical sensors

Only available columns are used. This avoids errors if a downloaded file uses a slightly different column set.

In [5]:
# 5. Select numerical sensors

planned_numerical_sensors = [
    "TP2",
    "TP3",
    "H1",
    "DV_pressure",
    "Reservoirs",
    "Oil_temperature",
    "Motor_current",
    "Caudal_impulses"
]

available_sensors = [col for col in planned_numerical_sensors if col in df.columns]

if len(available_sensors) == 0:
    raise ValueError("None of the planned numerical sensor columns were found.")

# Convert selected sensors to numeric in case they were loaded as object/string.
for col in available_sensors:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("Available numerical sensors used for feature engineering:")
for sensor in available_sensors:
    print("-", sensor)

Available numerical sensors used for feature engineering:
- TP2
- TP3
- H1
- DV_pressure
- Reservoirs
- Oil_temperature
- Motor_current
- Caudal_impulses


## 5. Create pressure-difference features

These features capture relationships between related pressure sensors.

In [6]:
# 6. Create pressure-difference features

df_features = df.copy()

pressure_diff_specs = {
    "TP2_TP3_diff": ("TP2", "TP3"),
    "Reservoirs_DV_pressure_diff": ("Reservoirs", "DV_pressure"),
    "TP3_Reservoirs_diff": ("TP3", "Reservoirs")
}

created_pressure_features = []

for new_col, (left_col, right_col) in pressure_diff_specs.items():
    if left_col in df_features.columns and right_col in df_features.columns:
        df_features[new_col] = df_features[left_col] - df_features[right_col]
        created_pressure_features.append(new_col)

print("Created pressure-difference features:")
for feature in created_pressure_features:
    print("-", feature)

Created pressure-difference features:
- TP2_TP3_diff
- Reservoirs_DV_pressure_diff
- TP3_Reservoirs_diff


## 6. Create rolling-window features

Rolling features are calculated using past observations only.

Windows used:

- 15 minutes: short-term instability
- 60 minutes: slower drift

For each selected sensor, this creates:

- rolling mean
- rolling standard deviation
- rolling minimum
- rolling maximum

In [7]:
# 7. Create rolling-window features

# Use timestamp as index for time-based rolling windows.
df_features = df_features.set_index(timestamp_col)

rolling_windows = {
    "15min": "15min",
    "60min": "60min"
}

created_rolling_features = []

for sensor in available_sensors:
    for window_name, window_value in rolling_windows.items():
        roll_obj = df_features[sensor].rolling(window=window_value, min_periods=1)

        feature_mean = f"{sensor}_roll_mean_{window_name}"
        feature_std = f"{sensor}_roll_std_{window_name}"
        feature_min = f"{sensor}_roll_min_{window_name}"
        feature_max = f"{sensor}_roll_max_{window_name}"

        df_features[feature_mean] = roll_obj.mean()
        df_features[feature_std] = roll_obj.std()
        df_features[feature_min] = roll_obj.min()
        df_features[feature_max] = roll_obj.max()

        created_rolling_features.extend([
            feature_mean,
            feature_std,
            feature_min,
            feature_max
        ])

# First row of each rolling std can be NaN because standard deviation needs more than one value.
print("Number of rolling features created:", len(created_rolling_features))
print(created_rolling_features[:10])

Number of rolling features created: 64
['TP2_roll_mean_15min', 'TP2_roll_std_15min', 'TP2_roll_min_15min', 'TP2_roll_max_15min', 'TP2_roll_mean_60min', 'TP2_roll_std_60min', 'TP2_roll_min_60min', 'TP2_roll_max_60min', 'TP3_roll_mean_15min', 'TP3_roll_std_15min']


## 7. Create lag features

Lag features provide previous sensor values at fixed time offsets.

Lags used:

- 5 minutes
- 15 minutes
- 60 minutes

The implementation uses `merge_asof` to match each timestamp with the latest available observation at or before the target lag time. This is safer than simple row shifting if timestamp spacing is irregular.

In [ ]:
# 8. Create time-based lag features

def add_time_lag_features(data, timestamp_index_name, sensor_cols, lag_minutes):
    """
    Add lag features using time-aware backward matching.

    For each timestamp t, the lag value is the latest available sensor value
    at or before t - lag_minutes.
    """
    working = data.reset_index().copy()
    working = working.sort_values(timestamp_index_name)

    base_times = working[[timestamp_index_name]].copy()
    base_times["target_lag_time"] = base_times[timestamp_index_name] - pd.Timedelta(minutes=lag_minutes)
    base_times = base_times.sort_values("target_lag_time")

    sensor_lookup = working[[timestamp_index_name] + sensor_cols].copy()
    sensor_lookup = sensor_lookup.sort_values(timestamp_index_name)

    matched = pd.merge_asof(
        base_times,
        sensor_lookup,
        left_on="target_lag_time",
        right_on=timestamp_index_name,
        direction="backward",
        suffixes=("", "_matched")
    )

    # matched contains timestamp_col_x as original time and timestamp_col_y as matched time.
    # Keep original timestamp and matched sensor values.
    result = working[[timestamp_index_name]].copy()

    matched_sensor_values = matched[sensor_cols].copy()
    matched_sensor_values.index = result.index

    for sensor in sensor_cols:
        result[f"{sensor}_lag_{lag_minutes}min"] = matched_sensor_values[sensor]

    result = result.set_index(timestamp_index_name)
    return result

lag_minutes_list = [5, 15, 60]
created_lag_features = []

for lag_minutes in lag_minutes_list:
    lag_df = add_time_lag_features(
        data=df_features,
        timestamp_index_name=timestamp_col,
        sensor_cols=available_sensors,
        lag_minutes=lag_minutes
    )

    for sensor in available_sensors:
        lag_col = f"{sensor}_lag_{lag_minutes}min"
        df_features[lag_col] = lag_df[lag_col]
        created_lag_features.append(lag_col)

print("Number of lag features created:", len(created_lag_features))
print(created_lag_features[:10])

## 8. Create rate-of-change features

Rate of change is calculated as the current sensor value minus the 5-minute lagged value.

In [ ]:
# 9. Create rate-of-change features

created_rate_features = []

for sensor in available_sensors:
    lag_col = f"{sensor}_lag_5min"
    if lag_col in df_features.columns:
        rate_col = f"{sensor}_rate_change_5min"
        df_features[rate_col] = df_features[sensor] - df_features[lag_col]
        created_rate_features.append(rate_col)

print("Number of rate-of-change features created:", len(created_rate_features))
print(created_rate_features)

## 9. Restore timestamp column and inspect engineered dataset

In [ ]:
# 10. Restore timestamp as a normal column

df_features = df_features.reset_index()

print("Original cleaned shape:", df.shape)
print("Engineered feature dataset shape:", df_features.shape)

new_feature_count = df_features.shape[1] - df.shape[1]
print("New features added:", new_feature_count)

df_features.head()

## 10. Check missing values after feature engineering

Lag features naturally create missing values at the beginning of the dataset because there is no earlier data available.

In [ ]:
# 11. Missing values after feature engineering

features_missing_values = df_features.isnull().sum().reset_index()
features_missing_values.columns = ["column", "missing_count"]
features_missing_values["missing_percentage"] = (
    features_missing_values["missing_count"] / len(df_features)
) * 100

features_missing_values = features_missing_values.sort_values(
    by="missing_count", ascending=False
)

features_missing_values.to_csv(
    OUTPUT_TABLES_DIR / "features_missing_values.csv",
    index=False
)

features_missing_values.head(20)

## 11. Plot missing values in engineered features

In [ ]:
# 12. Plot top missing values after feature engineering

top_missing = features_missing_values[features_missing_values["missing_count"] > 0].head(20)

if not top_missing.empty:
    plt.figure(figsize=(12, 6))
    plt.bar(top_missing["column"], top_missing["missing_percentage"])
    plt.xticks(rotation=90)
    plt.ylabel("Missing percentage")
    plt.title("Top missing values after feature engineering")
    plt.tight_layout()
    plt.savefig(OUTPUT_FIGURES_DIR / "feature_missing_values.png", dpi=300)
    plt.show()
else:
    print("No missing values found after feature engineering.")

## 12. Create feature list table

In [ ]:
# 13. Create feature list table

feature_records = []

for col in df_features.columns:
    if col == timestamp_col:
        feature_type = "timestamp"
    elif col in available_sensors:
        feature_type = "raw_sensor"
    elif col in created_pressure_features:
        feature_type = "pressure_difference"
    elif col in created_rolling_features:
        feature_type = "rolling_window"
    elif col in created_lag_features:
        feature_type = "lag"
    elif col in created_rate_features:
        feature_type = "rate_of_change"
    else:
        feature_type = "other_original_or_digital"

    feature_records.append({
        "feature_name": col,
        "feature_type": feature_type
    })

feature_list = pd.DataFrame(feature_records)
feature_list.to_csv(OUTPUT_TABLES_DIR / "feature_list.csv", index=False)

feature_list["feature_type"].value_counts()

## 13. Save feature engineering summary

In [ ]:
# 14. Save feature engineering summary

feature_engineering_summary = pd.DataFrame({
    "item": [
        "Input cleaned rows",
        "Input cleaned columns",
        "Output feature rows",
        "Output feature columns",
        "New features added",
        "Numerical sensors used",
        "Rolling windows",
        "Lag intervals",
        "Pressure difference features created",
        "Rate-of-change features created",
        "Total missing values after feature engineering"
    ],
    "value": [
        df.shape[0],
        df.shape[1],
        df_features.shape[0],
        df_features.shape[1],
        new_feature_count,
        ", ".join(available_sensors),
        "15min, 60min",
        "5min, 15min, 60min",
        ", ".join(created_pressure_features),
        ", ".join(created_rate_features),
        int(df_features.isnull().sum().sum())
    ]
})

feature_engineering_summary.to_csv(
    OUTPUT_TABLES_DIR / "feature_engineering_summary.csv",
    index=False
)

feature_engineering_summary

## 14. Save engineered feature dataset

The engineered dataset can be large. CSV is saved for compatibility with the next notebook. If needed, a Parquet version can be used later for speed.

In [ ]:
# 15. Save engineered dataset

df_features.to_csv(features_path, index=False)

print("Feature-engineered dataset saved to:")
print(features_path)
print("Final shape:", df_features.shape)

## 15. Completion checklist

In [ ]:
# 16. Completion checklist

completion_checklist = pd.DataFrame({
    "task": [
        "Cleaned dataset loaded",
        "Timestamp parsed and sorted",
        "Pressure-difference features created",
        "15-minute rolling features created",
        "60-minute rolling features created",
        "5-minute lag features created",
        "15-minute lag features created",
        "60-minute lag features created",
        "5-minute rate-of-change features created",
        "Feature list saved",
        "Feature engineering summary saved",
        "Engineered dataset saved"
    ],
    "status": [
        "Complete",
        "Complete",
        "Complete" if len(created_pressure_features) > 0 else "Check required",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete",
        "Complete"
    ]
})

completion_checklist.to_csv(
    OUTPUT_TABLES_DIR / "feature_engineering_completion_checklist.csv",
    index=False
)

completion_checklist

## 16. Confirm saved outputs

In [ ]:
# 17. Confirm saved files

print("Saved tables:")
for file in sorted(OUTPUT_TABLES_DIR.glob("*.csv")):
    if "feature" in file.name or "features" in file.name:
        print("-", file.name)

print("\nSaved figures:")
for file in sorted(OUTPUT_FIGURES_DIR.glob("*.png")):
    if "feature" in file.name:
        print("-", file.name)

print("\nProcessed dataset files:")
for file in sorted(PROCESSED_DATA_DIR.glob("*features*")):
    print("-", file.name)